# 02 Migration job: plays the Dataproc PySpark job

At the client, the migration path is STS → GCS → Dataproc → BigQuery. This notebook squeezes that into one step:

1. Read the legacy Delta tables (from notebook 01).
2. Apply the migration changes, including some **defects planted on purpose**.
3. Write the results to the BigQuery emulator, in dataset `local-project.sales`.

The defects come from `arf.demo` (`MISSING_ORDER_IDS` and the others), so ARF should report exactly what the demo reports.

In [ ]:
from datetime import date, datetime
from decimal import Decimal

from google.cloud import bigquery
from pyspark.sql import functions as F

import lab_common as lab
from arf.demo import CHANGED_AMOUNT_IDS, CHANGED_STATUS_IDS, MISSING_ORDER_IDS, NULLED_CUSTOMER_IDS

spark = lab.get_spark()      # registers the Delta tables that notebook 01 created
client = lab.bq_client()     # BigQuery client pointed at the emulator
print("Legacy tables:", [r.tableName for r in spark.sql("SHOW TABLES IN legacy_sales").collect()])

## 1. Transform: customers (a clean migration)

- The IST text timestamp becomes a real UTC `TIMESTAMP`. This change is agreed, and ARF allows it in `customers.yaml`.
- `credit_limit` gets 1e-9 of floating-point noise, the kind a different engine produces. ARF's tolerance should accept it.

In [ ]:
customers = (
    spark.table("legacy_sales.customers")
    .withColumn("created_ts", F.to_utc_timestamp(F.to_timestamp("created_ts", "yyyy-MM-dd HH:mm:ss"), "Asia/Kolkata"))
    .withColumn("credit_limit", F.col("credit_limit") + F.lit(1e-9))
)
customers.show(3, truncate=False)

## 2. Transform: orders (with planted defects)

Each line marked `DEFECT` is something ARF must catch. The status change is the interesting one: it leaves every count, SUM, MIN and MAX unchanged, so only the `row_hash` check can find it.

In [ ]:
orders = (
    spark.table("legacy_sales.orders")
    .filter(~F.col("order_id").isin(sorted(MISSING_ORDER_IDS)))                 # DEFECT: 3 rows lost
    .withColumnRenamed("cust_id", "customer_id")                                 # agreed rename (column_mapping)
    .withColumn("customer_id",
                F.when(F.col("order_id").isin(sorted(NULLED_CUSTOMER_IDS)), F.lit(None).cast("bigint"))
                 .otherwise(F.col("customer_id")))                               # DEFECT: 4 customer ids lost
    .withColumn("quantity", F.col("quantity").cast("string"))                    # DEFECT: INT became STRING
    .withColumn("amount",
                F.when(F.col("order_id").isin(sorted(CHANGED_AMOUNT_IDS)), F.col("amount") + F.lit(5))
                 .otherwise(F.col("amount")).cast("decimal(18,2)"))              # DEFECT: 2 amounts +5.00
    .withColumn("status",
                F.when(F.col("order_id").isin(sorted(CHANGED_STATUS_IDS)),
                       F.when(F.col("status") == "CANCELLED", F.lit("NEW")).otherwise(F.lit("CANCELLED")))
                 .otherwise(F.col("status")))                                    # DEFECT: 2 statuses changed
    .drop("legacy_flag")                                                         # DEFECT: column dropped
    .withColumn("load_ts", F.current_timestamp())                                # new audit column
)
print("orders after migration:", orders.count(), "rows")

## 3. Load into the BigQuery emulator

The tables are created with an explicit BigQuery schema, as a real pipeline would. Two more defects sit in that schema: `order_id` loses its NOT NULL, and `unit_price` becomes NUMERIC, which keeps 9 decimals instead of 12.

Rows go in through `insert_rows_json` (streaming inserts), which the emulator supports. Inventory is deliberately **not** migrated.

In [ ]:
SCHEMAS = {
    "customers": [
        bigquery.SchemaField("customer_id", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("customer_name", "STRING"),
        bigquery.SchemaField("signup_date", "DATE"),
        bigquery.SchemaField("created_ts", "TIMESTAMP"),
        bigquery.SchemaField("credit_limit", "FLOAT"),
        bigquery.SchemaField("lifetime_value", "NUMERIC"),
    ],
    "orders": [
        bigquery.SchemaField("order_id", "INTEGER"),            # DEFECT: NOT NULL lost
        bigquery.SchemaField("customer_id", "INTEGER"),
        bigquery.SchemaField("order_ts", "TIMESTAMP"),
        bigquery.SchemaField("quantity", "STRING"),
        bigquery.SchemaField("unit_price", "NUMERIC"),          # DEFECT: scale 12 -> 9
        bigquery.SchemaField("amount", "NUMERIC"),
        bigquery.SchemaField("discount", "FLOAT"),
        bigquery.SchemaField("status", "STRING"),
        bigquery.SchemaField("load_ts", "TIMESTAMP"),
    ],
}


def to_json(value):
    """insertAll takes JSON: decimals as text (no precision loss), dates and timestamps as ISO text."""
    if isinstance(value, Decimal):
        return str(value)
    if isinstance(value, datetime):            # naive values are UTC in this container (TZ=UTC)
        return value.strftime("%Y-%m-%d %H:%M:%S.%f")
    if isinstance(value, date):
        return value.isoformat()
    return value


def load(df, table):
    table_id = lab.bq_table(table)
    client.delete_table(table_id, not_found_ok=True)          # re-runs start clean
    client.create_table(bigquery.Table(table_id, schema=SCHEMAS[table]))
    rows = [{k: to_json(v) for k, v in r.asDict().items()} for r in df.collect()]
    for start in range(0, len(rows), 500):                    # send in batches of 500
        errors = client.insert_rows_json(table_id, rows[start:start + 500])
        if errors:
            raise RuntimeError(f"{table}: {errors[:3]}")
    print(f"{table_id}: {len(rows)} rows loaded")


client.create_dataset(f"{lab.BQ_PROJECT}.{lab.BQ_DATASET}", exists_ok=True)
load(customers, "customers")
load(orders, "orders")
client.delete_table(lab.bq_table("inventory"), not_found_ok=True)   # DEFECT: inventory never migrated

## 4. Check the BigQuery side

In [ ]:
for table in ("customers", "orders"):
    n = list(client.query(f"SELECT COUNT(*) AS n FROM `{lab.bq_table(table)}`").result())[0].n
    print(f"{lab.bq_table(table)}: {n} rows")